# L03 · Rigid-Body Physics and Stable Simulation

This lab moves from contact phenomena to numerical evidence. You will compare controlled time-discretization cases, measure contact with several signals, and then test how both sides of a contact pair affect sliding. The scenes use only built-in Boxes; no asset download or rendering is required.

## Before you run

GPU is the preferred backend for this course. Leave `ROBO_GENESIS_BACKEND=auto` unchanged; on the verified AMD ROCm environment, each runner selects `gs.amdgpu` and reports the actual backend used.

If no verified GPU is available, the same experiment can run on CPU as a slower compatibility fallback. Set `ROBO_GENESIS_BACKEND=cpu` only when you explicitly need that fallback or want to reproduce the CPU reference path. Every case runs in its own process because Genesis initialization is process-wide; this notebook kernel never calls `gs.init()`.

Generated `.npz` files and figures go below `ROBO_GENESIS_OUTPUTS_DIR` when that variable is set, or the repository `outputs/` directory otherwise. The runner reports its requested and actual backend. If an AMD run starts and then fails, the error remains visible rather than being replaced with a CPU result.

In [ ]:
import importlib.metadata as package_metadata
import os
import shlex
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from robo_genesis.course_utils import notebook_mode


def installed_version(distribution):
    try:
        return package_metadata.version(distribution)
    except package_metadata.PackageNotFoundError:
        return "not installed"


def load_case(path):
    with np.load(path, allow_pickle=False) as archive:
        return {name: archive[name].copy() for name in archive.files}


def scalar(case, name):
    return case[name].item()


def run_process(command, label):
    print("$ " + shlex.join(command), flush=True)
    completed = subprocess.run(command, text=True, capture_output=True, check=False)
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="", file=sys.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"{label} failed with return code {completed.returncode}")
    return completed


backend_mode = os.environ.get("ROBO_GENESIS_BACKEND", "auto").strip().lower()
if backend_mode not in {"auto", "cpu"}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")

runtime = notebook_mode("l03-rigid-body-physics", show_viewer=False)
output_dir = runtime["output_dir"]
contact_dir = (output_dir / "contact").resolve()
friction_dir = (output_dir / "friction").resolve()
contact_dir.mkdir(parents=True, exist_ok=True)
friction_dir.mkdir(parents=True, exist_ok=True)

environment = {
    "python": sys.version.split()[0],
    "genesis_world": installed_version("genesis-world"),
    "torch": installed_version("torch"),
    "requested_backend": backend_mode,
    "output_dir": str(output_dir.resolve()),
}
for key, value in environment.items():
    print(f"{key:>20}: {value}")

## Part A · Predict before simulating

All four cases use the same Box, table, density, friction, initial pose, seed, precision, and 1.5 s simulated duration. Only `dt` and `substeps` change. Before running the next cells, write down your answers:

1. Which cases will contain 150 outer samples, and which will contain 75?
2. At a fixed `dt`, what do you expect when `substeps` increases?
3. N1 and N4 share the same `substep_dt`. Should their states agree at shared sample times, and should their complete arrays have the same length?
4. If penetration decreases but a brief separation appears, can either number alone establish that the case is more stable?

In [ ]:
# Physical inputs are written here so you can inspect the experiment directly.
TABLE_CENTER_Z = 0.70
TABLE_HEIGHT = 0.05
TABLE_SIZE = (0.90, 0.60, TABLE_HEIGHT)
TABLE_POSITION = (0.35, 0.0, TABLE_CENTER_Z)
TABLE_FRICTION = 0.80
CUBE_SIZE = 0.08
CUBE_INITIAL_POSITION = (0.35, 0.0, 1.0)
CUBE_DENSITY = 500.0
CUBE_FRICTION = 0.50
EXPECTED_CENTER_Z = TABLE_CENTER_Z + TABLE_HEIGHT / 2 + CUBE_SIZE / 2
SIM_DURATION = 1.5


def validated_step_count(duration, dt):
    if duration <= 0 or dt <= 0:
        raise ValueError("duration and dt must be positive")
    n_steps = round(duration / dt)
    if n_steps < 1 or not np.isclose(n_steps * dt, duration, rtol=0.0, atol=1e-12):
        raise ValueError("duration must be an integer multiple of dt")
    return n_steps

CASE_SPECS = [
    {"label": "N1", "dt": 0.01, "substeps": 1},
    {"label": "N2", "dt": 0.01, "substeps": 2},
    {"label": "N3", "dt": 0.02, "substeps": 1},
    {"label": "N4", "dt": 0.02, "substeps": 2},
]
CONTACT_SCENE = {
    "table_size_m": TABLE_SIZE,
    "table_position_m": TABLE_POSITION,
    "table_friction": TABLE_FRICTION,
    "cube_size_m": (CUBE_SIZE,) * 3,
    "cube_initial_position_m": CUBE_INITIAL_POSITION,
    "cube_density_kg_m3": CUBE_DENSITY,
    "cube_friction": CUBE_FRICTION,
    "expected_resting_center_z_m": EXPECTED_CENTER_Z,
}
print("Fixed Part A scene:")
for name, value in CONTACT_SCENE.items():
    print(f"  {name}: {value}")

configuration_rows = []
for spec in CASE_SPECS:
    n_steps = validated_step_count(SIM_DURATION, spec["dt"])
    simulated_time = n_steps * spec["dt"]
    substep_dt = spec["dt"] / spec["substeps"]
    internal_updates = n_steps * spec["substeps"]
    assert np.isclose(simulated_time, SIM_DURATION)
    configuration_rows.append(
        {**spec, "n_steps": n_steps, "simulated_time": simulated_time,
         "substep_dt": substep_dt, "internal_updates": internal_updates}
    )

assert configuration_rows[0]["n_steps"] == configuration_rows[1]["n_steps"] == 150
assert configuration_rows[2]["n_steps"] == configuration_rows[3]["n_steps"] == 75
assert np.isclose(configuration_rows[0]["substep_dt"], configuration_rows[3]["substep_dt"])
assert configuration_rows[0]["n_steps"] != configuration_rows[3]["n_steps"]

lines = [
    "| Case | dt [s] | substeps | substep_dt [s] | samples | internal updates | duration [s] |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for row in configuration_rows:
    lines.append(
        "| {label} | {dt:.3f} | {substeps} | {substep_dt:.4f} | {n_steps} | "
        "{internal_updates} | {simulated_time:.1f} |".format(**row)
    )
display(Markdown("\n".join(lines)))

### Run the four isolated contact cases

The printed commands expose every changed argument. Each runner saves configuration, backend and version metadata, sampled state, contact evidence, and derived metrics in one `.npz` file. A failed child process prints its complete stdout and stderr before this notebook stops.

In [ ]:
CONTACT_FIELDS = {
    "requested_backend", "actual_backend", "genesis_version", "torch_version", "torch_hip",
    "seed", "precision", "dt", "substeps", "substep_dt", "duration",
    "n_steps", "table_size", "table_position", "table_friction",
    "cube_size", "cube_initial_position", "cube_density", "cube_friction", "time",
    "z", "vz", "contact_count", "expected_center_z", "first_contact_time",
    "penetration", "zero_contact_duration", "real_separation_duration",
    "max_rebound_clearance", "max_upward_vz", "contact_presence_transitions",
    "contact_count_changes", "settling_error", "initial_cube_pos", "final_cube_pos",
}

contact_results = {}
contact_commands = {}
for spec in CASE_SPECS:
    label = spec["label"]
    output_path = contact_dir / f"{label.lower()}.npz"
    command = [
        sys.executable, "-m", "robo_genesis.experiments.rigid_contact",
        "--backend", backend_mode,
        "--dt", str(spec["dt"]),
        "--substeps", str(spec["substeps"]),
        "--duration", str(SIM_DURATION),
        "--output", str(output_path),
    ]
    contact_commands[label] = command
    run_process(command, f"contact case {label}")
    case = load_case(output_path)
    missing = CONTACT_FIELDS - set(case)
    if missing:
        raise KeyError(f"contact case {label} is missing fields: {sorted(missing)}")
    contact_results[label] = case

print("PASS — all four contact cases completed in isolated processes")

### Read the metric definitions before the results

The runner samples once after every outer `scene.step()`. Event-time resolution is therefore the outer `dt`, not the internal time step; events shorter than `dt` may be missed.

`Case`, `backend`, `dt`, `substeps`, `internal dt`, and `samples` are direct configuration or runtime fields introduced above. The table below covers only derived or easily misread diagnostics.

| Diagnostic output column | Definition in this experiment | Interpretation and limitation |
|---|---|---|
| `penetration proxy [mm]` | `max(0, ideal resting center z − minimum sampled center z)` | A sampled center-height proxy, not exact continuous solver penetration. |
| `geometric separation [ms]` | Post-contact samples with zero reported contacts and cube-bottom clearance above `1e-5 m`, multiplied by `dt` | Stronger separation evidence than contact count alone, but still quantized by `dt`. |
| `max upward vz [m/s]` | Largest positive vertical velocity after first contact | Shows upward motion; combine it with geometric separation and height history before calling it rebound. |
| `contact-count changes` | Number of adjacent post-contact samples whose reported contact counts differ | Indicates contact-manifold variability, not a standalone stability score; it also depends on sampling cadence. |
| `settling error [mm]` | Absolute difference between ideal resting height and mean center height over the final 0.2 s | Smaller means the tail mean is closer to ideal; the mean can still hide oscillation. |

Suggested reading order: compare `internal dt` with the penetration proxy first, then inspect geometric separation and the height trajectory, and use contact-count changes only as supporting evidence. No single column is a complete stability verdict.

In [ ]:
def first_true_index(mask):
    indices = np.flatnonzero(mask)
    return int(indices[0]) if indices.size else -1


def contact_metrics_from_samples(case):
    dt = scalar(case, "dt")
    z = case["z"]
    vz = case["vz"]
    contact_count = case["contact_count"]
    expected_center_z = scalar(case, "expected_center_z")
    contact_mask = contact_count > 0
    first_index = first_true_index(contact_mask)
    first_time = case["time"][first_index] if first_index >= 0 else np.nan
    penetration = max(0.0, expected_center_z - float(np.min(z)))

    if first_index >= 0:
        post_contact = contact_mask[first_index:]
        post_counts = contact_count[first_index:]
        clearance = z[first_index:] - CUBE_SIZE / 2 - (TABLE_CENTER_Z + TABLE_HEIGHT / 2)
        separation = (~post_contact) & (clearance > 1e-5)
        zero_contact_duration = float(np.count_nonzero(~post_contact) * dt)
        real_separation_duration = float(np.count_nonzero(separation) * dt)
        max_rebound_clearance = max(0.0, float(np.max(clearance)))
        max_upward_vz = max(0.0, float(np.max(vz[first_index:])))
        contact_presence_transitions = int(np.count_nonzero(np.diff(post_contact.astype(int))))
        contact_count_changes = int(np.count_nonzero(np.diff(post_counts)))
    else:
        zero_contact_duration = np.nan
        real_separation_duration = np.nan
        max_rebound_clearance = np.nan
        max_upward_vz = np.nan
        contact_presence_transitions = 0
        contact_count_changes = 0

    tail_samples = max(1, round(0.2 / dt))
    settling_error = abs(float(np.mean(z[-tail_samples:])) - expected_center_z)
    return {
        "first_contact_time": first_time,
        "penetration": penetration,
        "zero_contact_duration": zero_contact_duration,
        "real_separation_duration": real_separation_duration,
        "max_rebound_clearance": max_rebound_clearance,
        "max_upward_vz": max_upward_vz,
        "contact_presence_transitions": contact_presence_transitions,
        "contact_count_changes": contact_count_changes,
        "settling_error": settling_error,
    }


contact_rows = []
contact_data_checks = {}
contact_metrics = {}
for spec in CASE_SPECS:
    label = spec["label"]
    case = contact_results[label]
    expected_samples = round(SIM_DURATION / spec["dt"])
    arrays_ok = all(
        case[name].shape == (expected_samples,) and np.isfinite(case[name]).all()
        for name in ("time", "z", "vz", "contact_count")
    )
    metadata_ok = (
        scalar(case, "requested_backend") == backend_mode
        and scalar(case, "actual_backend") in {"cpu", "amdgpu"}
        and np.isclose(scalar(case, "dt"), spec["dt"])
        and int(scalar(case, "substeps")) == spec["substeps"]
        and int(scalar(case, "n_steps")) == expected_samples
        and np.isclose(scalar(case, "duration"), SIM_DURATION)
        and np.allclose(case["table_size"], TABLE_SIZE)
        and np.allclose(case["table_position"], TABLE_POSITION)
        and np.isclose(scalar(case, "table_friction"), TABLE_FRICTION)
        and np.isclose(scalar(case, "cube_size"), CUBE_SIZE)
        and np.allclose(case["cube_initial_position"], CUBE_INITIAL_POSITION)
        and np.isclose(scalar(case, "cube_density"), CUBE_DENSITY)
        and np.isclose(scalar(case, "cube_friction"), CUBE_FRICTION)
    )
    metrics = contact_metrics_from_samples(case)
    contact_metrics[label] = metrics
    for name, recomputed in metrics.items():
        saved = scalar(case, name)
        if np.isnan(recomputed):
            assert np.isnan(saved), name
        else:
            assert np.isclose(saved, recomputed), (name, saved, recomputed)
    contact_observed = np.isfinite(metrics["first_contact_time"])
    not_through_table = float(np.min(case["z"])) > TABLE_CENTER_Z
    contact_data_checks[label] = arrays_ok and metadata_ok and contact_observed and not_through_table
    contact_rows.append({
        "label": label,
        "backend": scalar(case, "actual_backend"),
        "dt": scalar(case, "dt"),
        "substeps": int(scalar(case, "substeps")),
        "substep_dt": scalar(case, "substep_dt"),
        "samples": len(case["time"]),
        "penetration_mm": 1000 * metrics["penetration"],
        "separation_ms": 1000 * metrics["real_separation_duration"],
        "max_upward_vz": metrics["max_upward_vz"],
        "contact_changes": int(metrics["contact_count_changes"]),
        "settling_error_mm": 1000 * metrics["settling_error"],
    })

lines = [
    "| Case | backend | dt [s] | substeps | internal dt [s] | samples | penetration proxy [mm] | geometric separation [ms] | max upward vz [m/s] | contact-count changes | settling error [mm] |",
    "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
]
for row in contact_rows:
    lines.append(
        "| {label} | {backend} | {dt:.3f} | {substeps} | {substep_dt:.4f} | {samples} | "
        "{penetration_mm:.3f} | {separation_ms:.1f} | {max_upward_vz:.4f} | "
        "{contact_changes} | {settling_error_mm:.4f} |".format(**row)
    )
display(Markdown("\n".join(lines)))
assert all(contact_data_checks.values()), contact_data_checks

In [ ]:
CONTACT_COLORS = {"N1": "#247BA0", "N2": "#2A9D5B", "N3": "#D95F43", "N4": "#7555A6"}


def draw_contact_state(axis, cube_position, title, cube_color):
    axis.add_patch(plt.Rectangle(
        (0.35 - 0.9 / 2, TABLE_CENTER_Z - TABLE_HEIGHT / 2),
        0.9, TABLE_HEIGHT, color="#8C6138", label="fixed table",
    ))
    axis.add_patch(plt.Rectangle(
        (cube_position[0] - CUBE_SIZE / 2, cube_position[2] - CUBE_SIZE / 2),
        CUBE_SIZE, CUBE_SIZE, color=cube_color, label="dynamic cube",
    ))
    axis.set(xlim=(-0.18, 0.88), ylim=(0.62, 1.08), xlabel="x [m]", ylabel="z [m]")
    axis.set_title(title)
    axis.set_aspect("equal")
    axis.grid(alpha=0.2)
    axis.legend(fontsize=8)


n1 = contact_results["N1"]
state_figure, state_axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True, sharey=True)
draw_contact_state(state_axes[0], n1["initial_cube_pos"], "Initial state", "#2A9D8F")
draw_contact_state(state_axes[1], n1["final_cube_pos"], "Final state", "#E76F51")
state_figure.suptitle("State-derived schematic — not a Genesis camera render")
state_figure.tight_layout()
contact_state_path = output_dir / "contact_initial_final.png"
state_figure.savefig(contact_state_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(state_figure)

trajectory_figure, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
for spec in CASE_SPECS:
    label = spec["label"]
    case = contact_results[label]
    legend = f"{label}: dt={spec['dt']}, substeps={spec['substeps']}"
    axes[0].plot(case["time"], 1000 * (case["z"] - scalar(case, "expected_center_z")), color=CONTACT_COLORS[label], label=legend)
    axes[1].plot(case["time"], case["vz"], color=CONTACT_COLORS[label], label=legend)
    axes[2].step(case["time"], case["contact_count"], where="post", color=CONTACT_COLORS[label], label=legend)
axes[0].axhline(0.0, color="#444444", linestyle="--", linewidth=1)
axes[0].set(ylabel="center-height error [mm]", title="Height relative to ideal rest")
axes[1].axhline(0.0, color="#444444", linewidth=1)
axes[1].set(ylabel="vertical velocity [m/s]", title="Vertical motion")
axes[2].set(xlabel="simulated time [s]", ylabel="contact count", title="Sampled contact evidence")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
trajectory_figure.tight_layout()
contact_trajectory_path = output_dir / "contact_trajectories.png"
trajectory_figure.savefig(contact_trajectory_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(trajectory_figure)
print("saved:", contact_state_path.resolve())
print("saved:", contact_trajectory_path.resolve())

In [ ]:
def shared_sample_indices(left_time, right_time, decimals=12):
    left = np.asarray(left_time, dtype=float)
    right = np.asarray(right_time, dtype=float)
    if left.ndim != 1 or right.ndim != 1 or left.size == 0 or right.size == 0:
        raise ValueError("timelines must be non-empty 1-D arrays")
    if np.any(np.diff(left) <= 0) or np.any(np.diff(right) <= 0):
        raise ValueError("timelines must be strictly increasing")
    _, left_indices, right_indices = np.intersect1d(
        np.round(left, decimals), np.round(right, decimals), return_indices=True
    )
    if left_indices.size == 0:
        raise ValueError("timelines have no common sample timestamps")
    return left_indices, right_indices


n1, n2, n3, n4 = (contact_results[label] for label in ("N1", "N2", "N3", "N4"))
n1_indices, n4_indices = shared_sample_indices(n1["time"], n4["time"])
shared_times = n1["time"][n1_indices]
assert np.allclose(shared_times, n4["time"][n4_indices])
shared_z_difference = float(np.max(np.abs(n1["z"][n1_indices] - n4["z"][n4_indices])))
shared_vz_difference = float(np.max(np.abs(n1["vz"][n1_indices] - n4["vz"][n4_indices])))
SHARED_Z_TOLERANCE = 1e-5
SHARED_VZ_TOLERANCE = 1e-4
penetrations = {label: contact_metrics[label]["penetration"] for label in contact_results}
part_a_checks = {
    **{f"{label}_data_valid": passed for label, passed in contact_data_checks.items()},
    "N1_to_N2_penetration_decreased": penetrations["N2"] < penetrations["N1"],
    "N3_to_N4_penetration_decreased": penetrations["N4"] < penetrations["N3"],
    "N1_N4_shared_z_within_tolerance": shared_z_difference <= SHARED_Z_TOLERANCE,
    "N1_N4_shared_vz_within_tolerance": shared_vz_difference <= SHARED_VZ_TOLERANCE,
}

separation_lines = []
for label in ("N1", "N2", "N3", "N4"):
    milliseconds = 1000 * contact_metrics[label]["real_separation_duration"]
    separation_lines.append(f"- {label}: observed geometric separation = {milliseconds:.1f} ms")

display(Markdown(
    "### Evidence-based interpretation\n\n"
    f"- N1→N2 penetration proxy: {1000 * penetrations['N1']:.3f} → {1000 * penetrations['N2']:.3f} mm.\n"
    f"- N3→N4 penetration proxy: {1000 * penetrations['N3']:.3f} → {1000 * penetrations['N4']:.3f} mm.\n"
    f"- N1/N4 share {len(shared_times)} timestamps; maximum aligned differences are "
    f"{1000 * shared_z_difference:.6f} mm in z and {shared_vz_difference:.6g} m/s in vz.\n"
    + "\n".join(separation_lines)
    + "\n\nA smaller penetration proxy is one piece of evidence, not a complete stability verdict. "
    "Interpret it together with separation, velocity, contact changes, settling error, and the full trajectories."
))
print(part_a_checks)

## Part B · Friction and sustained stopping

Two identical free cubes settle on one table, then receive the same `2.0 m/s` horizontal velocity. The baseline changes only cube friction. Genesis 1.3.3 uses the larger of the two rigid-body friction values after the default runtime ratio, so predict before running:

1. What are the two effective pair coefficients when table friction is 0.50 and cube friction is 0.10 or 0.80?
2. Which lane should travel farther before sustained stopping?
3. Why must the runner record angular velocity as well as `x(t)` and `vx(t)`?
4. Would swapping the orange and blue Surface colors change either trajectory?

In [ ]:
# Part B keeps geometry and density fixed while changing material friction.
FRICTION_TABLE_SIZE = (2.0, 0.8, TABLE_HEIGHT)
FRICTION_TABLE_POSITION = (0.0, 0.0, TABLE_CENTER_Z)
FRICTION_CUBE_SIZE = 0.08
FRICTION_CUBE_DENSITY = 500.0
RESTING_CENTER_Z = TABLE_CENTER_Z + TABLE_HEIGHT / 2 + FRICTION_CUBE_SIZE / 2
START_X = -0.60
LOW_LANE_Y = -0.15
HIGH_LANE_Y = 0.15


def effective_pair_friction(table_friction, cube_friction):
    if table_friction < 0 or cube_friction < 0:
        raise ValueError("friction values must be non-negative")
    return max(table_friction, cube_friction)


FRICTION_SPEC = {
    "dt": 0.01,
    "substeps": 2,
    "settle_duration": 0.30,
    "measure_duration": 2.00,
    "initial_vx": 2.00,
    "table_friction": 0.50,
    "low_friction": 0.10,
    "high_friction": 0.80,
    "stop_speed": 0.01,
    "stop_hold": 0.10,
}
FRICTION_SCENE = {
    "table_size_m": FRICTION_TABLE_SIZE,
    "table_position_m": FRICTION_TABLE_POSITION,
    "cube_size_m": (FRICTION_CUBE_SIZE,) * 3,
    "cube_density_kg_m3": FRICTION_CUBE_DENSITY,
    "start_x_m": START_X,
    "lane_y_m": (LOW_LANE_Y, HIGH_LANE_Y),
    "resting_center_z_m": RESTING_CENTER_Z,
}
print("Fixed Part B scene:")
for name, value in FRICTION_SCENE.items():
    print(f"  {name}: {value}")
print("Part B baseline inputs:")
for name, value in FRICTION_SPEC.items():
    print(f"  {name}: {value}")


def run_friction_case(spec, output_name):
    output_path = friction_dir / output_name
    command = [
        sys.executable, "-m", "robo_genesis.experiments.rigid_friction",
        "--backend", backend_mode,
        "--dt", str(spec["dt"]),
        "--substeps", str(spec["substeps"]),
        "--settle-duration", str(spec["settle_duration"]),
        "--measure-duration", str(spec["measure_duration"]),
        "--initial-vx", str(spec["initial_vx"]),
        "--table-friction", str(spec["table_friction"]),
        "--low-friction", str(spec["low_friction"]),
        "--high-friction", str(spec["high_friction"]),
        "--stop-speed", str(spec["stop_speed"]),
        "--stop-hold", str(spec["stop_hold"]),
        "--output", str(output_path),
    ]
    run_process(command, f"friction case {output_name}")
    return load_case(output_path)


baseline_friction = run_friction_case(FRICTION_SPEC, "baseline.npz")
print("effective low lane:", effective_pair_friction(0.50, 0.10))
print("effective high lane:", effective_pair_friction(0.50, 0.80))

In [ ]:
def sustained_stop_index(speed, threshold, hold_samples):
    speed = np.asarray(speed, dtype=float)
    if speed.ndim != 1 or speed.size == 0 or not np.isfinite(speed).all():
        raise ValueError("speed must be a non-empty, finite 1-D array")
    if threshold < 0 or hold_samples < 1:
        raise ValueError("threshold must be non-negative and hold_samples positive")
    below_threshold = np.abs(speed) < threshold
    for index in range(len(speed) - hold_samples + 1):
        if np.all(below_threshold[index:index + hold_samples]):
            return index
    return -1


def stop_measurement(case, suffix):
    hold_samples = int(scalar(case, "hold_samples"))
    stop_index = sustained_stop_index(
        case[f"vx_{suffix}"], scalar(case, "stop_speed"), hold_samples
    )
    if stop_index < 0:
        return {"index": -1, "time": np.nan, "distance": np.nan}
    return {
        "index": stop_index,
        "time": float(case["time"][stop_index]),
        "distance": float(case[f"x_{suffix}"][stop_index] - case[f"x_{suffix}"][0]),
    }


FRICTION_FIELDS = {
    "requested_backend", "actual_backend", "genesis_version", "torch_version", "torch_hip",
    "seed", "precision", "dt", "substeps", "substep_dt",
    "settle_duration", "measure_duration", "settle_steps", "measure_steps",
    "initial_vx", "table_friction", "low_friction", "high_friction",
    "effective_low_friction", "effective_high_friction", "stop_speed",
    "stop_hold", "hold_samples", "table_size", "table_position",
    "cube_size", "cube_density", "resting_center_z", "low_lane_y",
    "high_lane_y", "start_x", "start_low", "start_high", "final_low", "final_high",
    "stop_time_low", "stop_time_high", "stop_distance_low", "stop_distance_high",
}
missing_friction_fields = FRICTION_FIELDS - set(baseline_friction)
if missing_friction_fields:
    raise KeyError(f"baseline friction result is missing fields: {sorted(missing_friction_fields)}")
FRICTION_ARRAY_FIELDS = (
    "time", "x_low", "x_high", "vx_low", "vx_high",
    "omega_y_low", "omega_y_high", "contacts_low", "contacts_high",
)
expected_measure_samples = round(FRICTION_SPEC["measure_duration"] / FRICTION_SPEC["dt"]) + 1
for name in FRICTION_ARRAY_FIELDS:
    values = baseline_friction[name]
    assert values.shape == (expected_measure_samples,), (name, values.shape)
    assert np.isfinite(values).all(), name
baseline_stop_measurements = {
    suffix: stop_measurement(baseline_friction, suffix) for suffix in ("low", "high")
}
for suffix, measurement in baseline_stop_measurements.items():
    for field in ("time", "distance"):
        saved = scalar(baseline_friction, f"stop_{field}_{suffix}")
        recomputed = measurement[field]
        if np.isnan(recomputed):
            assert np.isnan(saved), (suffix, field)
        else:
            assert np.isclose(saved, recomputed), (suffix, field, saved, recomputed)
assert scalar(baseline_friction, "requested_backend") == backend_mode
assert scalar(baseline_friction, "actual_backend") in {"cpu", "amdgpu"}
for name, expected in FRICTION_SPEC.items():
    assert np.isclose(scalar(baseline_friction, name), expected), name
assert np.isclose(scalar(baseline_friction, "effective_low_friction"), 0.50)
assert np.isclose(scalar(baseline_friction, "effective_high_friction"), 0.80)

def format_measurement(value, digits=4):
    return f"{value:.{digits}f}" if np.isfinite(value) else "not stopped in window"

friction_rows = []
for label, suffix in (("Low-μ cube", "low"), ("High-μ cube", "high")):
    friction_rows.append({
        "label": label,
        "cube": scalar(baseline_friction, f"{suffix}_friction"),
        "table": scalar(baseline_friction, "table_friction"),
        "effective": scalar(baseline_friction, f"effective_{suffix}_friction"),
        "stop_time": baseline_stop_measurements[suffix]["time"],
        "stop_distance": baseline_stop_measurements[suffix]["distance"],
    })
lines = [
    "| Lane | cube μ | table μ | effective pair μ | sustained stop [s] | stop distance [m] |",
    "|---|---:|---:|---:|---:|---:|",
]
for row in friction_rows:
    lines.append(
        f"| {row['label']} | {row['cube']:.2f} | {row['table']:.2f} | "
        f"{row['effective']:.2f} | {format_measurement(row['stop_time'], 3)} | "
        f"{format_measurement(row['stop_distance'])} |"
    )
display(Markdown("\n".join(lines)))

def draw_friction_state(axis, low_position, high_position, title):
    axis.add_patch(plt.Rectangle((-1.0, -0.4), 2.0, 0.8, color="#8C6138"))
    for position, color, label in (
        (low_position, "#F27A24", "low μ"),
        (high_position, "#2674C8", "high μ"),
    ):
        axis.add_patch(plt.Rectangle(
            (position[0] - 0.04, position[1] - 0.04), 0.08, 0.08, color=color, label=label
        ))
    axis.set(xlim=(-1.05, 1.05), ylim=(-0.45, 0.45), xlabel="x [m]", ylabel="y [m]")
    axis.set_aspect("equal")
    axis.set_title(title)
    axis.legend(fontsize=8)

friction_figure, friction_axes = plt.subplots(2, 2, figsize=(12, 8))
draw_friction_state(friction_axes[0, 0], baseline_friction["start_low"], baseline_friction["start_high"], "Initial state")
draw_friction_state(friction_axes[0, 1], baseline_friction["final_low"], baseline_friction["final_high"], "Final state")
time = baseline_friction["time"]
for suffix, color, label in (("low", "#F27A24", "low μ"), ("high", "#2674C8", "high μ")):
    displacement = baseline_friction[f"x_{suffix}"] - baseline_friction[f"x_{suffix}"][0]
    friction_axes[1, 0].plot(time, displacement, color=color, label=f"{label} x")
    friction_axes[1, 0].plot(time, baseline_friction[f"vx_{suffix}"], linestyle="--", color=color, label=f"{label} vx")
    friction_axes[1, 1].plot(time, baseline_friction[f"omega_y_{suffix}"], color=color, label=label)
friction_axes[1, 0].set(xlabel="time [s]", ylabel="x [m] or vx [m/s]", title="Displacement and linear velocity")
friction_axes[1, 1].set(xlabel="time [s]", ylabel="angular velocity y [rad/s]", title="Rotation during sliding")
for axis in friction_axes[1]:
    axis.axhline(0.0, color="#444444", linewidth=1)
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
friction_figure.suptitle("Measured state and trajectories — not camera images")
friction_figure.tight_layout()
friction_baseline_path = output_dir / "friction_baseline.png"
friction_figure.savefig(friction_baseline_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(friction_figure)
print("saved:", friction_baseline_path.resolve())

## One-factor exercise · Change the table, not the cubes

The next run changes table friction from 0.50 to 0.30 and nothing else. Before executing it, predict which effective pair coefficient changes, which lane travels farther, and which lane should remain approximately unchanged. The comparison checks directions and a declared tolerance; it does not require one hard-coded stopping distance.

In [ ]:
MODIFIED_TABLE_FRICTION = 0.30
modified_spec = dict(FRICTION_SPEC)
modified_spec["table_friction"] = MODIFIED_TABLE_FRICTION
controlled_fields = set(FRICTION_SPEC) - {"table_friction"}
assert all(modified_spec[name] == FRICTION_SPEC[name] for name in controlled_fields)
modified_friction = run_friction_case(modified_spec, "table_friction_030.npz")
missing_modified_fields = FRICTION_FIELDS - set(modified_friction)
if missing_modified_fields:
    raise KeyError(f"modified friction result is missing fields: {sorted(missing_modified_fields)}")
assert scalar(modified_friction, "requested_backend") == backend_mode
for name, expected in modified_spec.items():
    assert np.isclose(scalar(modified_friction, name), expected), name

for name in FRICTION_ARRAY_FIELDS:
    values = modified_friction[name]
    assert values.shape == (expected_measure_samples,), (name, values.shape)
    assert np.isfinite(values).all(), name
modified_stop_measurements = {
    suffix: stop_measurement(modified_friction, suffix) for suffix in ("low", "high")
}
for suffix, measurement in modified_stop_measurements.items():
    for field in ("time", "distance"):
        saved = scalar(modified_friction, f"stop_{field}_{suffix}")
        recomputed = measurement[field]
        if np.isnan(recomputed):
            assert np.isnan(saved), (suffix, field)
        else:
            assert np.isclose(saved, recomputed), (suffix, field, saved, recomputed)
baseline_low_distance = baseline_stop_measurements["low"]["distance"]
baseline_high_distance = baseline_stop_measurements["high"]["distance"]
modified_low_distance = modified_stop_measurements["low"]["distance"]
modified_high_distance = modified_stop_measurements["high"]["distance"]
HIGH_LANE_UNCHANGED_TOLERANCE = 1e-3
distances_finite = bool(np.isfinite([
    baseline_low_distance, baseline_high_distance,
    modified_low_distance, modified_high_distance,
]).all())
if distances_finite:
    friction_direction_checks = {
        "baseline_low_lane_travels_farther": baseline_low_distance > baseline_high_distance,
        "modified_low_lane_travels_farther": modified_low_distance > baseline_low_distance,
        "high_lane_approximately_unchanged": (
            abs(modified_high_distance - baseline_high_distance) <= HIGH_LANE_UNCHANGED_TOLERANCE
        ),
    }
else:
    friction_direction_checks = {
        "baseline_low_lane_travels_farther": False,
        "modified_low_lane_travels_farther": False,
        "high_lane_approximately_unchanged": False,
    }
part_b_checks = {
    "all_stop_distances_finite": distances_finite,
    **friction_direction_checks,
    "low_effective_pair_changed": np.isclose(scalar(modified_friction, "effective_low_friction"), 0.30),
    "high_effective_pair_unchanged": np.isclose(scalar(modified_friction, "effective_high_friction"), 0.80),
}

comparison_lines = [
    "| Lane | baseline effective μ | modified effective μ | baseline stop [m] | modified stop [m] | change [m] |",
    "|---|---:|---:|---:|---:|---:|",
]
for label, suffix in (("Low-μ cube", "low"), ("High-μ cube", "high")):
    baseline_distance = baseline_stop_measurements[suffix]["distance"]
    modified_distance = modified_stop_measurements[suffix]["distance"]
    comparison_lines.append(
        f"| {label} | {scalar(baseline_friction, f'effective_{suffix}_friction'):.2f} | "
        f"{scalar(modified_friction, f'effective_{suffix}_friction'):.2f} | "
        f"{format_measurement(baseline_distance)} | {format_measurement(modified_distance)} | "
        f"{format_measurement(modified_distance - baseline_distance)} |"
    )
display(Markdown("\n".join(comparison_lines)))

comparison_figure, comparison_axis = plt.subplots(figsize=(10, 4.5))
for result, style, table_label in (
    (baseline_friction, "-", "table μ=0.50"),
    (modified_friction, "--", "table μ=0.30"),
):
    comparison_axis.plot(result["time"], result["vx_low"], style, color="#F27A24", label=f"low-μ cube, {table_label}")
    comparison_axis.plot(result["time"], result["vx_high"], style, color="#2674C8", label=f"high-μ cube, {table_label}")
comparison_axis.axhline(0.0, color="#444444", linewidth=1)
comparison_axis.set(xlabel="time [s]", ylabel="vx [m/s]", title="Only table friction changes")
comparison_axis.grid(alpha=0.25)
comparison_axis.legend(fontsize=8)
comparison_figure.tight_layout()
friction_exercise_path = output_dir / "friction_table_comparison.png"
comparison_figure.savefig(friction_exercise_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(comparison_figure)
print(part_b_checks)
print("saved:", friction_exercise_path.resolve())

## Checkpoint before L04

Use your generated tables and plots to answer these questions in your own words:

- Why do N1 and N4 have the same internal timestep but different sample counts?
- Which combination of signals supports or weakens a claim of stable contact?
- Why is `contact_count == 0` not sufficient evidence of rebound?
- How does the Genesis 1.3.3 pair rule explain the asymmetric result of the table-friction exercise?
- Which conclusions are specific to this scene, engine version, backend, and observation window?

In [ ]:
backend_evidence = {
    scalar(case, "actual_backend")
    for case in (*contact_results.values(), baseline_friction, modified_friction)
}
backend_checks = {
    "one_actual_backend_used": len(backend_evidence) == 1,
    "forced_cpu_honored": backend_mode != "cpu" or backend_evidence == {"cpu"},
}
all_checks = {**part_a_checks, **part_b_checks, **backend_checks}
failed_checks = [name for name, passed in all_checks.items() if not passed]
for name, passed in all_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
print("requested backend:", backend_mode)
print("actual backend:", sorted(backend_evidence))
print("evidence directory:", output_dir.resolve())
if failed_checks:
    raise AssertionError("L03 checks failed: " + ", ".join(failed_checks))
print("L03 CHECK: PASSED")